# Explore nightly radiance as image arrays

Use the project's **`.venv` kernel** (`requirements-notebooks.txt`). This notebook
only loads data; it performs no modeling, clustering, or time-series analysis.

Three NumPy arrays have axes **(time, row, column)**, or **(night, height, width)**:

- `a2_cube`: QA-screened, non-gap-filled A2 corrected radiance on the native grid (~500 m).
- `reallocation_500m_cube`: reallocated radiance aggregated to exactly that A2 grid.
- `reallocation_10m_cube`: the actual nightly 10 m allocation outputs, retained before aggregation.

All spatial indices are zero-based integers. Rows run top to bottom and columns
left to right; no latitude/longitude labels are used for indexing. Every grid cell
is retained, including unavailable cells as NaN. Zero radiance remains zero.
The native inputs preserve their separate valid-observation masks.

The 10 m array is **memory-mapped and read-only** (~6.5 GB on disk), so opening it
does not load the whole cube into RAM. It includes the original processing margin
outside Dyer. `native_aoi_mask_10m` identifies fine pixels overlapping the selected
native county cells, but is not applied automatically. The 10 m grid is projected,
so its dimensions are not exactly 50 times the A2 grid and its row/column indices
do not directly align with A2. All dates use the same ordering.

These fine pixels are a structural allocation proxy, not independent 10 m
measurements or validated fine-scale outages. Missingness can increase under the
complete-kernel rule; native aggregation adds a complete-footprint rule. The saved
fine scenes were checked to reproduce the published native aggregates exactly.
Dates are product day labels, not exact local overpass times.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts/run_reallocation_check.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Launch this notebook from within ntl-outage-detection.")

DATA_DIR = ROOT / "data/published/reallocation_check"
FINE_DIR = DATA_DIR / "fine"
# Optional: scans the entire 6.5 GB file, which may be slow over a mounted drive.
VERIFY_FINE_HASHES = False

units = "nW cm^-2 sr^-1"

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

In [2]:
# Load the published native-grid data and restore its rectangular scene.
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
if manifest.get("status") != "complete":
    raise ValueError("The native comparison run is not complete.")
array_path = DATA_DIR / "comparison_arrays.npz"
if sha256(array_path) != manifest["output_sha256"][array_path.name]:
    raise ValueError("Native array checksum mismatch.")

with np.load(array_path, allow_pickle=False) as saved:
    dates = pd.DatetimeIndex(pd.to_datetime(saved["dates"]), name="date")
    county_mask = saved["county"].astype(bool)
    a2_flat = saved["direct"].copy()
    reallocation_flat = saved["allocated"].copy()

assert dates.is_unique and dates.is_monotonic_increasing
assert a2_flat.shape == reallocation_flat.shape == (len(dates), int(county_mask.sum()))
height, width = county_mask.shape

def restore_scene(flat):
    cube = np.full((len(dates), height, width), np.nan, dtype=flat.dtype)
    cube[:, county_mask] = flat
    return cube

a2_cube = restore_scene(a2_flat)
reallocation_500m_cube = restore_scene(reallocation_flat)
np.testing.assert_equal(a2_cube[:, county_mask], a2_flat)
np.testing.assert_equal(reallocation_500m_cube[:, county_mask], reallocation_flat)
del a2_flat, reallocation_flat

# Optional paired mask for later use; neither input is restricted to it here.
common_valid_500m = np.isfinite(a2_cube) & np.isfinite(reallocation_500m_cube)

In [3]:
# Open retained 10 m scenes without loading every night into memory.
fine_manifest_path = FINE_DIR / "manifest.json"
if not fine_manifest_path.exists():
    raise FileNotFoundError(
        "Fine scenes have not been exported. Run from the repository root: "
        ".venv/bin/python scripts/export_reallocation_fine.py"
    )
fine_manifest = json.loads(fine_manifest_path.read_text())
if fine_manifest.get("status") != "complete":
    raise ValueError("The fine-resolution export is incomplete.")
if fine_manifest["identity"]["comparison_manifest_sha256"] != sha256(DATA_DIR / "manifest.json"):
    raise ValueError("Fine and native outputs are from different comparison runs.")
np.testing.assert_array_equal(pd.to_datetime(fine_manifest["dates"]), dates)

if VERIFY_FINE_HASHES:
    for name, expected_hash in fine_manifest["output_sha256"].items():
        if sha256(FINE_DIR / name) != expected_hash:
            raise ValueError(f"Fine export checksum mismatch: {name}")

reallocation_10m_cube = np.load(FINE_DIR / "reallocation_10m.npy", mmap_mode="r", allow_pickle=False)
native_aoi_mask_10m = np.load(FINE_DIR / "native_aoi_mask.npy", mmap_mode="r", allow_pickle=False)
assert reallocation_10m_cube.shape == tuple(fine_manifest["shape"])
assert reallocation_10m_cube.shape[0] == len(dates)
assert reallocation_10m_cube.dtype == np.dtype("float32")
assert native_aoi_mask_10m.shape == reallocation_10m_cube.shape[1:]
assert native_aoi_mask_10m.dtype == np.dtype("bool")

print(f"Loaded {len(dates)} nights: {dates[0].date()} through {dates[-1].date()}")
for name, cube in (("A2", a2_cube), ("Reallocation ~500 m", reallocation_500m_cube),
                   ("Reallocation 10 m", reallocation_10m_cube)):
    print(f"{name}: {cube.shape} (time, height, width), {cube.dtype}")

Loaded 72 nights: 2025-02-05 through 2025-04-17
A2: (72, 81, 139) (time, height, width), float64
Reallocation ~500 m: (72, 81, 139) (time, height, width), float64
Reallocation 10 m: (72, 4097, 5533) (time, height, width), float32


In [4]:
# Optional 2D pandas dataframes use integer row/column indices.
rows = pd.RangeIndex(height, name="row")
columns = pd.RangeIndex(width, name="column")
a2_frames = [pd.DataFrame(scene.copy(), index=rows, columns=columns) for scene in a2_cube]
reallocation_500m_frames = [
    pd.DataFrame(scene.copy(), index=rows, columns=columns)
    for scene in reallocation_500m_cube
]
a2_by_date = dict(zip(dates.strftime("%Y-%m-%d"), a2_frames))
reallocation_500m_by_date = dict(zip(dates.strftime("%Y-%m-%d"), reallocation_500m_frames))

def fine_frame(night):
    """One 10 m scene as an integer-indexed dataframe; night is an index or date."""
    t = int(night) if isinstance(night, (int, np.integer)) else dates.get_loc(pd.Timestamp(night))
    scene = reallocation_10m_cube[t]
    return pd.DataFrame(
        scene,
        index=pd.RangeIndex(scene.shape[0], name="row"),
        columns=pd.RangeIndex(scene.shape[1], name="column"),
        copy=False,
    )

# Retain names from the first version of this notebook for convenience.
reallocation_cube = reallocation_500m_cube
reallocation_frames = reallocation_500m_frames
reallocation_by_date = reallocation_500m_by_date

## Ready to explore

- `a2_cube[t, row, col]`: one A2 cell; `a2_cube[t]`: its full scene.
- `reallocation_500m_cube[t, row, col]`: the matching native-grid allocation cell.
- `reallocation_10m_cube[t, fine_row, fine_col]`: one retained fine-grid proxy cell.
- `a2_by_date["2025-04-08"]` or `reallocation_500m_by_date["2025-04-08"]`: native dataframe by date.
- `fine_frame("2025-04-08")`: a fine-grid dataframe, created on demand.

Native dataframes are editable copies; changes do not update the cubes. Fine frames
initially share the read-only mapped data; use `.copy()` when you need an editable
copy. Avoid copying the whole fine cube unless you want to load all ~6.5 GB into RAM.

For a later image plot, use `plt.imshow(a2_cube[t], origin="upper")` after importing
`matplotlib.pyplot as plt`; row and column numbers are the image coordinates.
You can mask a selected fine scene for display with
`np.where(native_aoi_mask_10m, reallocation_10m_cube[t], np.nan)`.
No geocoding is required to index or display any of these arrays.

In [5]:
NIGHT = "2025-04-08"
t = dates.get_loc(pd.Timestamp(NIGHT))
a2_night = a2_cube[t]
reallocation_500m_night = reallocation_500m_cube[t]
reallocation_10m_night = reallocation_10m_cube[t]  # mapped view, not a full-cube copy

# Preview dataframe structure only; scene-edge entries may be NaN.
display(a2_frames[t].iloc[:5, :5])
display(reallocation_500m_frames[t].iloc[:5, :5])
display(fine_frame(t).iloc[:5, :5])

column,0,1,2,3,4
row,,,,,
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN


column,0,1,2,3,4
row,,,,,
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN


column,0,1,2,3,4
row,,,,,
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN


## Make a simple movie

`array_movie(cube, dates=dates)` creates an inline movie with play/pause, a frame
slider, and playback-speed controls. It uses integer row/column coordinates and
a **fixed color scale across all nights**. Missing cells are gray; an entirely
missing night is retained and labeled “no valid observations.” Each frame gets
equal screen time, even if the supplied dates have gaps.

This helper is intended for the small native-grid datasets. It rejects oversized
inputs before scanning their values, including the full 10 m cube. No ffmpeg or
additional packages are required. The examples below use the same color limits
for both methods; no smoothing, imputation, or statistical analysis is applied.
Run these cells to create the movies. JavaScript notebook output must be trusted
and supported by your notebook viewer.

To select a time window, slice both inputs together, e.g.
`array_movie(a2_cube[56:], dates=dates[56:], title="A2 event window")`.
The returned HTML can optionally be saved with
`Path("a2_movie.html").write_text(a2_movie.data, encoding="utf-8")`.

In [6]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def array_movie(cube, dates=None, *, title="Nightly radiance", fps=3,
                vmin=None, vmax=None, cmap="magma", colorbar_label=units):
    """Return a playable HTML movie of a small (time, height, width) array.

    Also accepts a list of equally shaped 2D arrays/dataframes. Color limits
    default to the finite minimum/maximum across the entire supplied sequence.
    Pass the same vmin/vmax to multiple movies for direct visual comparison.
    """
    shape = np.shape(cube)
    if len(shape) != 3 or any(size == 0 for size in shape):
        raise ValueError("Expected a nonempty (time, height, width) array.")
    if shape[1] * shape[2] > 250_000 or np.prod(shape) > 20_000_000:
        raise ValueError(
            "This simple movie helper is limited to small scenes. "
            "Use a2_cube or reallocation_500m_cube, not the full 10 m cube."
        )
    if not np.isfinite(fps) or fps <= 0:
        raise ValueError("fps must be a positive finite number.")
    if dates is not None and len(dates) != shape[0]:
        raise ValueError("Supply exactly one date label per frame.")
    labels = ([str(value)[:10] for value in dates] if dates is not None
              else [f"Frame {i}" for i in range(shape[0])])
    values = np.asarray(cube, dtype=float)
    finite = values[np.isfinite(values)]
    low = float(finite.min()) if finite.size else 0.0
    high = float(finite.max()) if finite.size else 1.0
    lower = low if vmin is None else float(vmin)
    upper = high if vmax is None else float(vmax)
    if not np.isfinite(lower) or not np.isfinite(upper) or lower > upper:
        raise ValueError("Color limits must be finite with vmin <= vmax.")
    if lower == upper:
        # A constant or all-zero sequence still needs a usable color scale.
        upper = lower + max(abs(lower) * 0.01, 1.0)
    colors = plt.get_cmap(cmap).copy()
    colors.set_bad("#bdbdbd")
    fig, ax = plt.subplots(figsize=(7, 5), dpi=90)
    im = ax.imshow(np.ma.masked_invalid(values[0]), origin="upper",
                   interpolation="nearest", cmap=colors, vmin=lower, vmax=upper)
    ax.set(xlabel="Column", ylabel="Row")
    fig.colorbar(im, ax=ax, label=colorbar_label)
    heading = ax.set_title("")
    fig.tight_layout()

    def update(i):
        frame = values[i]
        im.set_data(np.ma.masked_invalid(frame))
        missing = " — no valid observations" if not np.isfinite(frame).any() else ""
        heading.set_text(f"{title} | {labels[i]} | {i + 1}/{len(values)}{missing}")
        return im, heading

    animation = FuncAnimation(fig, update, frames=len(values), interval=1000 / fps,
                              repeat=True, blit=False, cache_frame_data=False)
    try:
        return HTML(animation.to_jshtml(fps=fps, default_mode="loop"))
    finally:
        plt.close(fig)  # Prevent an extra static plot and release figure resources.


In [7]:
# One shared full-range scale, held fixed across time and both movies.
movie_vmin = min(float(np.nanmin(a2_cube)), float(np.nanmin(reallocation_500m_cube)))
movie_vmax = max(float(np.nanmax(a2_cube)), float(np.nanmax(reallocation_500m_cube)))

a2_movie = array_movie(a2_cube, dates=dates, title="A2", fps=3,
                       vmin=movie_vmin, vmax=movie_vmax)
display(a2_movie)

In [8]:
reallocation_movie = array_movie(
    reallocation_500m_cube, dates=dates, title="Reallocation (~500 m)", fps=3,
    vmin=movie_vmin, vmax=movie_vmax,
)
display(reallocation_movie)